# Lecture 6

## Gym

### 0. Install Package

In [ ]:
%pip install gymnasium gymnasium[all] pyvirtualdisplay ffmpeg stable-baselines3 "numpy==1.26.4"

### 1. Frozenlake-v1

In [ ]:
import base64
from IPython.display import HTML, display
import gymnasium as gym
from gymnasium.wrappers import RecordVideo

env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, reward_schedule=(1, -1, 0), render_mode= "rgb_array")
n_episodes = 10
env = RecordVideo(
    env,
    video_folder="./videos",
    episode_trigger=lambda episode: episode == 0,
    # Allow enough frames for all episodes, including reset frames.
    video_length=n_episodes * (env.spec.max_episode_steps + 1) + 1,
    name_prefix="all",
)

try:
    for _ in range(n_episodes):
        obs, info = env.reset()
        done = False

        while not done:
            action = env.action_space.sample()  # Replace with your policy.
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
finally:
    env.close()  # Saves videos/all-episode-0.mp4

def embed_video(video_path):
    video = open(video_path, "rb").read()
    video_base64 = base64.b64encode(video).decode("utf-8")
    display(
        HTML(
            f'<video width="640" height="480" controls><source src="data:video/mp4;base64,{video_base64}" type="video/mp4"></video>'
        )
    )
# 播放录制的文件（请根据实际生成的文件名调整路径）
embed_video("./videos/all-episode-0.mp4")

### 2. Policy Gradient

In [ ]:
import torch
from torch.distributions import Categorical
import math
def collect_trajectory(env, policy, n_roullout = 10, gamma = 0.99):
    loss = 0.0
    success = 0
    for _ in range(n_roullout):
        state, _ = env.reset()
        done = False
        T = 0
        log_prob = 0.0
        while not done:
            dist = Categorical(logits=policy[state])
            action = dist.sample()
            log_prob += dist.log_prob(action)
            next_state, reward, terminated, truncated, _ = env.step(action.item())
            state = next_state
            done = terminated or truncated
            T = T + 1
        if reward > 0:
            success += 1
        reward = math.pow(gamma, T) * reward
        loss = loss - log_prob * reward
    return loss, success / n_roullout

torch.manual_seed(42)
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, reward_schedule=(1, -1, 0), render_mode = 'none')
env.reset(seed=42)
policy = torch.nn.Parameter(torch.zeros(16, 4))
optimizer = torch.optim.Adam([policy], lr=0.02)
for episode in range(1, 100):
    loss, success = collect_trajectory(env, policy, 100)
    print(f"Episode {episode}: loss = {loss.item():.4f}, success rate = {success:.2f}")
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
env.close()

In [ ]:
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, reward_schedule=(1, -1, 0), render_mode= "rgb_array")
n_episodes = 10
env = RecordVideo(
    env,
    video_folder="./trained_videos",
    episode_trigger=lambda episode: episode == 0,
    # Allow enough frames for all episodes, including reset frames.
    video_length=n_episodes * (env.spec.max_episode_steps + 1) + 1,
    name_prefix="FrozenLake",
)
try:
    for _ in range(n_episodes):
        obs, info = env.reset()
        done = False
        while not done:
            dist = Categorical(logits=policy[obs])
            action = dist.sample().item()
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
finally:
    env.close()  # Saves videos/all-episode-0.mp4
    
embed_video("./trained_videos/FrozenLake-episode-0.mp4")

## RL

### 0. LunarLander-v3

In [4]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecVideoRecorder
import gymnasium as gym
import base64
from IPython.display import HTML, display
import gymnasium as gym
from gymnasium.wrappers import RecordVideo


env = gym.make("LunarLander-v3", render_mode="rgb_array")
model = PPO("MlpPolicy", env, verbose=1, device = 'cpu')
n_episodes = 10
vec_env = model.get_env()
video_env = VecVideoRecorder(
    vec_env,
    video_folder="./videos",
    record_video_trigger=lambda step: step == 0,
    video_length=1000,
    name_prefix="LunarLander",
)
try:
    obs = video_env.reset()
    for _ in range(1000):
        actions, _ = model.predict(obs, deterministic=True)
        obs, rewards, dones, infos = video_env.step(actions)
finally:
    video_env.close()


def embed_video(video_path):
    video = open(video_path, "rb").read()
    video_base64 = base64.b64encode(video).decode("utf-8")
    display(
        HTML(
            f'<video width="640" height="480" controls><source src="data:video/mp4;base64,{video_base64}" type="video/mp4"></video>'
        )
    )


embed_video("./videos/LunarLander-step-0-to-step-1000.mp4")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Saving video to /home/bakasakib/Documents/ISDN_Robofab/Codes/videos/LunarLander-step-0-to-step-1000.mp4
MoviePy - Building video /home/bakasakib/Documents/ISDN_Robofab/Codes/videos/LunarLander-step-0-to-step-1000.mp4.
MoviePy - Writing video /home/bakasakib/Documents/ISDN_Robofab/Codes/videos/LunarLander-step-0-to-step-1000.mp4



MoviePy - Done !
MoviePy - video ready /home/bakasakib/Documents/ISDN_Robofab/Codes/videos/LunarLander-step-0-to-step-1000.mp4


In [5]:
%pip install tensorboard

Note: you may need to restart the kernel to use updated packages.


In [6]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecVideoRecorder
import gymnasium as gym

env = gym.make("LunarLander-v3", render_mode="rgb_array")
model = PPO("MlpPolicy", env, verbose=0, device = 'cpu', tensorboard_log="./lunar_log/")
model.learn(total_timesteps=100_000,  tb_log_name="PPO")

vec_env = model.get_env()
video_env = VecVideoRecorder(
    vec_env,
    video_folder="./trained_videos",
    record_video_trigger=lambda step: step == 0,
    video_length=1000,
    name_prefix="LunarLander",
)
try:
    obs = video_env.reset()
    for _ in range(1000):
        actions, _ = model.predict(obs, deterministic=True)
        obs, rewards, dones, infos = video_env.step(actions)
finally:
    video_env.close()

Saving video to /home/bakasakib/Documents/ISDN_Robofab/Codes/trained_videos/LunarLander-step-0-to-step-1000.mp4
MoviePy - Building video /home/bakasakib/Documents/ISDN_Robofab/Codes/trained_videos/LunarLander-step-0-to-step-1000.mp4.
MoviePy - Writing video /home/bakasakib/Documents/ISDN_Robofab/Codes/trained_videos/LunarLander-step-0-to-step-1000.mp4



MoviePy - Done !
MoviePy - video ready /home/bakasakib/Documents/ISDN_Robofab/Codes/trained_videos/LunarLander-step-0-to-step-1000.mp4


In [7]:
embed_video("./trained_videos/LunarLander-step-0-to-step-1000.mp4")

In [8]:
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv
train_env = make_vec_env("LunarLander-v3", n_envs=16, vec_env_cls=SubprocVecEnv, env_kwargs={"render_mode": None})
model = PPO("MlpPolicy", train_env, verbose=0, device="cpu", tensorboard_log="./lunar_log/",
             batch_size=64,
             n_steps=1024,
             learning_rate=3e-4,
             ent_coef=0.01,
             clip_range=0.2,
             gae_lambda=0.98,
             gamma=0.999)
model.learn(total_timesteps=1_000_000, tb_log_name="PPO")

In [9]:
vec_env = model.get_env()
video_env = VecVideoRecorder(
    vec_env,
    video_folder="./trained_videos",
    record_video_trigger=lambda step: step == 0,
    video_length=1000,
    name_prefix="LunarLander",
)
try:
    obs = video_env.reset()
    for _ in range(1000):
        actions, _ = model.predict(obs, deterministic=True)
        obs, rewards, dones, infos = video_env.step(actions)
finally:
    video_env.close()
embed_video("./trained_videos/LunarLander-step-0-to-step-1000.mp4")

AssertionError: The render_mode must be 'rgb_array', not None